In [1]:
# Backend
import os

os.environ["KERAS_BACKEND"] = "jax"

In [2]:
# Setup
import json
from importlib import import_module
from pathlib import Path
from typing import cast

from keras.utils import plot_model

from src.models import LearnableCutFlowModel
from src.utils import print

In [3]:
# Parameters
# Config
with open("1-dataset:config.json", "r") as f:
    dataset_config = json.load(f)

with open("2-model:config.json", "r") as f:
    model_config = json.load(f)

# Dataset
dataset_name = "mock1"
features = dataset_config[dataset_name]["features"]
n_samples = dataset_config[dataset_name]["n_samples"]
seed = dataset_config[dataset_name]["seed"]

# Model
model_name = "lcf_seq"  # * (lcf_par, lcf_seq)
model_builder = model_config[model_name]["model_builder"]
centers = model_config[model_name]["centers"][dataset_name]

# Path
prefix = f"2-model:{model_name}"
FIGURES_DIR = Path("figures")

In [4]:
# Dataset
module = import_module(f"src.datasets.{dataset_name}")
load_data = getattr(module, "load_data")
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(100000, 4)
y_train.shape=(100000, 1)
x_test.shape=(100000, 4)
y_test.shape=(100000, 1)


In [5]:
# Model
ModelBuilder = getattr(import_module("src.models"), model_builder)
model = ModelBuilder(x_train.shape, centers, features=features, name=model_name)
model = cast(LearnableCutFlowModel, model)
model.adapt(x_train)
model.compile(optimizer="adam", loss="crossentropy")

In [6]:
# Model structure
model.summary(print_fn=print)

plot_model(
    model,
    to_file=FIGURES_DIR / f"{prefix}-structure.pdf",
    show_shapes=True,
    show_layer_names=True,
    show_trainable=True,
)

Model: "lcf_seq"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, 4)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 4)         │          9 │ inputs[0][0]      │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ learnable_importan… │ (None, 4)         │          4 │ normalization[0]… │
│ (LearnableImportan… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ split (Split)       │ [(None, 1),       │          0 │ learnable_import… │
│                     │ (None, 1), (None, │            │   